# EA-TTM — Locked Final-Test Evaluation

**Project:** Extreme-Aware Tiny Time Mixers (EA-TTM) for Retail Demand Forecasting under Regime Shifts  
**Phase:** 10 — Single locked final-test evaluation  
**Scope:** Evaluate every pre-locked comparison model exactly once on `d_1914:d_1941`, persist final evidence, and prohibit post-test tuning.

## Development lock

Phase 09 fixed the primary model as:

$$
\boxed{
\text{Phase 08 best-balanced EA-TTM, epoch 15}
}
$$

No Phase 10 result is permitted to modify the model, threshold, feature contract, checkpoint choice, or training procedure.

The locked final-test interval is:

$$
d_{1914}:d_{1941}.
$$

## Pre-registered comparison set

The final evaluation includes only models whose parameters or rules were fixed **before** the final test was opened:

1. Weekly Seasonal Naive;
2. Zero-shot TTM;
3. Frozen HP-TTM;
4. Retail-only gated adapter;
5. Ungated cross-feed residual adapter;
6. Primary EA-TTM.

The Phase 08 best-overall diagnostic checkpoint is deliberately excluded from the final comparison because the Phase 09 development lock identifies the epoch-15 balanced checkpoint as the primary EA-TTM model.

## Evaluation metrics

For all $N$ final-test observations:

$$
\mathrm{MSE}
=
\frac{1}{N}
\sum_{i=1}^{N}
(y_i-\hat y_i)^2,
$$

$$
\mathrm{RMSE}
=
\sqrt{\mathrm{MSE}},
$$

$$
\mathrm{MAE}
=
\frac{1}{N}
\sum_{i=1}^{N}
|y_i-\hat y_i|,
$$

and:

$$
\mathrm{WAPE}
=
\frac{
\sum_{i=1}^{N}|y_i-\hat y_i|
}{
\sum_{i=1}^{N}|y_i|
}.
$$

The same locked Phase 07 shock definition is used to report:

- normal-regime MSE;
- all-extreme MSE;
- positive-extreme MSE;
- negative-extreme MSE.

## Paired bootstrap uncertainty

For the primary EA-TTM versus HP-TTM comparison, uncertainty is assessed with a paired bootstrap over the 30,490 store-item series.

For series $j$:

$$
\delta_j
=
\mathrm{MSE}_{HP,j}
-
\mathrm{MSE}_{EA,j}.
$$

A positive value favors EA-TTM.

The notebook uses 2,000 deterministic bootstrap resamples of the series-level paired differences and reports the 95% percentile interval:

$$
[
Q_{0.025},
Q_{0.975}
].
$$

This is an inferential diagnostic only. It does not alter any model decision.

In [1]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import math
import os
import random
import subprocess
import sys
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from torch import nn

## Project preflight

In [2]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/EA_TTM_RETAIL")

PRETRAINED_TTM_DIR = (
    PROJECT_ROOT
    / "models"
    / "pretrained_ttm"
)

HEAD_PROBED_TTM_DIR = (
    PROJECT_ROOT
    / "models"
    / "head_probed_ttm"
)

PRIMARY_ADAPTER_DIR = (
    PROJECT_ROOT
    / "models"
    / "ea_ttm_adapter"
)

PROCESSED_M5_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "m5"
)

METADATA_DIR = (
    PROJECT_ROOT
    / "metadata"
)

RESULTS_TABLE_DIR = (
    PROJECT_ROOT
    / "results"
    / "tables"
)

PHASE09_CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "runs"
    / "09_validation_ablation"
    / "checkpoints"
)

RUN_DIR = (
    PROJECT_ROOT
    / "runs"
    / "10_locked_final_test"
)

for directory in (
    RESULTS_TABLE_DIR,
    RUN_DIR,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

required_artifacts = {
    "pretrained_ttm_config": PRETRAINED_TTM_DIR / "config.json",
    "head_probed_ttm_config": HEAD_PROBED_TTM_DIR / "config.json",
    "primary_adapter_state": PRIMARY_ADAPTER_DIR / "adapter_state.pt",
    "primary_adapter_config": PRIMARY_ADAPTER_DIR / "adapter_config.json",
    "retail_only_checkpoint": PHASE09_CHECKPOINT_DIR / "retail_only_gated_best_state.pt",
    "ungated_checkpoint": PHASE09_CHECKPOINT_DIR / "ungated_crossfeed_best_state.pt",
    "split_manifest": METADATA_DIR / "02_SPLIT_MANIFEST.json",
    "phase05_alignment": METADATA_DIR / "05_DECODER_ALIGNMENT_WEIGHTS.npz",
    "phase06_scaler": METADATA_DIR / "06_FEATURE_SCALER.json",
    "phase06_contract": METADATA_DIR / "06_FEATURE_CONTRACT.json",
    "phase07_extreme_contract": METADATA_DIR / "07_EXTREME_TARGET_CONTRACT.json",
    "phase08_summary": METADATA_DIR / "08_RUN_SUMMARY.json",
    "phase09_summary": METADATA_DIR / "09_RUN_SUMMARY.json",
    "development_lock": METADATA_DIR / "09_MODEL_DEVELOPMENT_LOCK.json",
    "processed_sales": PROCESSED_M5_DIR / "sales_train_evaluation.parquet",
    "processed_calendar": PROCESSED_M5_DIR / "calendar.parquet",
    "processed_prices": PROCESSED_M5_DIR / "sell_prices.parquet",
}

preflight = {
    name: {
        "path": str(path),
        "exists": path.exists(),
    }
    for name, path in required_artifacts.items()
}

missing_artifacts = [
    item["path"]
    for item in preflight.values()
    if not item["exists"]
]

assert not missing_artifacts, (
    "Phase 10 prerequisites are incomplete: "
    + "; ".join(missing_artifacts)
)

Mounted at /content/drive


## IBM TSFM runtime

In [3]:
TSFM_DISTRIBUTION_VERSION = "0.3.9"
TSFM_GIT_REPOSITORY = "https://github.com/ibm-granite/granite-tsfm.git"
TSFM_GIT_TAG = f"v{TSFM_DISTRIBUTION_VERSION}"
TSFM_SOURCE_LOCK = METADATA_DIR / "00_TSFM_SOURCE_LOCK.txt"

RUNTIME_VENDOR_ROOT = Path("/content/ea_ttm_vendor")
TSFM_SOURCE_ROOT = RUNTIME_VENDOR_ROOT / "granite-tsfm"

RUNTIME_VENDOR_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

def module_available(
    module_name: str,
) -> bool:
    return (
        importlib.util.find_spec(
            module_name
        )
        is not None
    )

def run_command(
    command: list[str],
    cwd: Path | None = None,
) -> str:
    result = subprocess.run(
        command,
        cwd=(
            str(cwd)
            if cwd is not None
            else None
        ),
        check=True,
        capture_output=True,
        text=True,
    )

    return result.stdout.strip()

if not module_available(
    "tsfm_public"
):
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-deps",
            (
                "granite-tsfm=="
                f"{TSFM_DISTRIBUTION_VERSION}"
            ),
        ],
        check=True,
    )

    importlib.invalidate_caches()

if not module_available(
    "tsfm_public"
):
    if TSFM_SOURCE_LOCK.exists():
        tsfm_source_commit = (
            TSFM_SOURCE_LOCK
            .read_text(
                encoding="utf-8"
            )
            .strip()
        )
    else:
        tag_query = subprocess.run(
            [
                "git",
                "ls-remote",
                "--tags",
                TSFM_GIT_REPOSITORY,
                f"refs/tags/{TSFM_GIT_TAG}",
            ],
            check=False,
            capture_output=True,
            text=True,
        ).stdout.strip()

        if tag_query:
            tsfm_source_commit = (
                tag_query.split()[0]
            )
        else:
            tsfm_source_commit = (
                run_command(
                    [
                        "git",
                        "ls-remote",
                        TSFM_GIT_REPOSITORY,
                        "refs/heads/main",
                    ]
                )
                .split()[0]
            )

        TSFM_SOURCE_LOCK.write_text(
            tsfm_source_commit,
            encoding="utf-8",
        )

    if TSFM_SOURCE_ROOT.exists():
        subprocess.run(
            [
                "rm",
                "-rf",
                str(TSFM_SOURCE_ROOT),
            ],
            check=True,
        )

    run_command(
        [
            "git",
            "clone",
            "--quiet",
            "--no-checkout",
            TSFM_GIT_REPOSITORY,
            str(TSFM_SOURCE_ROOT),
        ]
    )

    run_command(
        [
            "git",
            "checkout",
            "--quiet",
            tsfm_source_commit,
        ],
        cwd=TSFM_SOURCE_ROOT,
    )

    sys.path.insert(
        0,
        str(TSFM_SOURCE_ROOT),
    )

    importlib.invalidate_caches()

assert module_available(
    "tsfm_public"
)

from tsfm_public.models.tinytimemixer.modeling_tinytimemixer import (
    TinyTimeMixerForPrediction,
)

from tsfm_public.toolkit.time_series_preprocessor import (
    DEFAULT_FREQUENCY_MAPPING,
)

## Verify the development lock before opening the final test

In [4]:
split_manifest = json.loads(
    (
        METADATA_DIR
        / "02_SPLIT_MANIFEST.json"
    ).read_text(
        encoding="utf-8"
    )
)

feature_scaler = json.loads(
    (
        METADATA_DIR
        / "06_FEATURE_SCALER.json"
    ).read_text(
        encoding="utf-8"
    )
)

feature_contract = json.loads(
    (
        METADATA_DIR
        / "06_FEATURE_CONTRACT.json"
    ).read_text(
        encoding="utf-8"
    )
)

extreme_contract = json.loads(
    (
        METADATA_DIR
        / "07_EXTREME_TARGET_CONTRACT.json"
    ).read_text(
        encoding="utf-8"
    )
)

phase08_summary = json.loads(
    (
        METADATA_DIR
        / "08_RUN_SUMMARY.json"
    ).read_text(
        encoding="utf-8"
    )
)

phase09_summary = json.loads(
    (
        METADATA_DIR
        / "09_RUN_SUMMARY.json"
    ).read_text(
        encoding="utf-8"
    )
)

development_lock = json.loads(
    (
        METADATA_DIR
        / "09_MODEL_DEVELOPMENT_LOCK.json"
    ).read_text(
        encoding="utf-8"
    )
)

primary_adapter_config = json.loads(
    (
        PRIMARY_ADAPTER_DIR
        / "adapter_config.json"
    ).read_text(
        encoding="utf-8"
    )
)

DATASET_SHA256 = (
    phase09_summary[
        "dataset_sha256"
    ]
)

SPLIT_SHA256 = (
    phase09_summary[
        "split_sha256"
    ]
)

DEVELOPMENT_LOCK_SHA256 = (
    phase09_summary[
        "development_lock_sha256"
    ]
)

CONTEXT_LENGTH = int(
    split_manifest["context_length"]
)

FORECAST_HORIZON = int(
    split_manifest["forecast_horizon"]
)

TEST_ORIGIN = int(
    split_manifest["test_origin"]
)

assert CONTEXT_LENGTH == 512
assert FORECAST_HORIZON == 28
assert TEST_ORIGIN == 1913

assert (
    split_manifest[
        "locked_test_interval"
    ]
    == ["d_1914", "d_1941"]
)

assert (
    development_lock[
        "primary_model"
    ]
    == "Phase 08 best-balanced EA-TTM"
)

assert int(
    development_lock[
        "primary_epoch"
    ]
) == 15

assert (
    development_lock[
        "post_lock_tuning_permitted"
    ]
    is False
)

assert (
    development_lock[
        "development_lock_sha256"
    ]
    == DEVELOPMENT_LOCK_SHA256
)

assert (
    phase09_summary[
        "post_phase09_tuning_permitted"
    ]
    is False
)

assert (
    primary_adapter_config[
        "selected_checkpoint"
    ]
    == "best_balanced"
)

assert int(
    primary_adapter_config[
        "selected_epoch"
    ]
) == 15

print(
    "MODEL DEVELOPMENT LOCK: VERIFIED"
)

print(
    "Primary model:",
    development_lock[
        "primary_model"
    ],
)

print(
    "Primary epoch:",
    development_lock[
        "primary_epoch"
    ],
)

print(
    "Final test:",
    development_lock[
        "locked_final_test_interval"
    ],
)

MODEL DEVELOPMENT LOCK: VERIFIED
Primary model: Phase 08 best-balanced EA-TTM
Primary epoch: 15
Final test: ['d_1914', 'd_1941']


## Reproducibility state

In [5]:
BASE_SEED = 2026

def set_global_seed(
    seed: int,
) -> None:
    os.environ[
        "PYTHONHASHSEED"
    ] = str(seed)

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed(
            seed
        )

        torch.cuda.manual_seed_all(
            seed
        )

    torch.use_deterministic_algorithms(
        True,
        warn_only=True,
    )

    if hasattr(
        torch.backends,
        "cudnn",
    ):
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

set_global_seed(
    BASE_SEED
)

assert torch.cuda.is_available(), (
    "CUDA GPU is required for Phase 10."
)

DEVICE = torch.device(
    "cuda"
)

FREQUENCY = "D"

FREQUENCY_TOKEN = int(
    DEFAULT_FREQUENCY_MAPPING[
        FREQUENCY
    ]
)

INFERENCE_BATCH_SIZE = 512

## Open the locked final-test data

This is the first development phase permitted to materialize `d_1914:d_1941`.

In [6]:
identifier_columns = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
]

context_start_day = (
    TEST_ORIGIN
    - CONTEXT_LENGTH
    + 1
)

context_columns = [
    f"d_{day}"
    for day in range(
        context_start_day,
        TEST_ORIGIN + 1,
    )
]

target_columns = [
    f"d_{day}"
    for day in range(
        TEST_ORIGIN + 1,
        TEST_ORIGIN
        + FORECAST_HORIZON
        + 1,
    )
]

test_frame = pd.read_parquet(
    (
        PROCESSED_M5_DIR
        / "sales_train_evaluation.parquet"
    ),
    columns=(
        identifier_columns
        + context_columns
        + target_columns
    ),
)

test_context = test_frame[
    context_columns
].to_numpy(
    dtype=np.float32,
    copy=True,
)

test_target = test_frame[
    target_columns
].to_numpy(
    dtype=np.float32,
    copy=True,
)

SERIES_COUNT = int(
    test_context.shape[0]
)

assert test_context.shape == (
    30_490,
    512,
)

assert test_target.shape == (
    30_490,
    28,
)

assert target_columns[0] == "d_1914"
assert target_columns[-1] == "d_1941"

print(
    "LOCKED FINAL TEST OPENED:",
    target_columns[0],
    "through",
    target_columns[-1],
)

LOCKED FINAL TEST OPENED: d_1914 through d_1941


## Weekly seasonal-naive forecast

In [7]:
last_observed_week = (
    test_context[
        :,
        -7:,
    ]
)

seasonal_naive_forecast = np.tile(
    last_observed_week,
    (
        1,
        math.ceil(
            FORECAST_HORIZON
            / 7
        ),
    ),
)[
    :,
    :FORECAST_HORIZON,
]

assert (
    seasonal_naive_forecast.shape
    == test_target.shape
)

## Frozen zero-shot and HP-TTM inference

In [8]:
zero_shot_ttm = (
    TinyTimeMixerForPrediction
    .from_pretrained(
        PRETRAINED_TTM_DIR
    )
    .to(DEVICE)
)

hp_ttm = (
    TinyTimeMixerForPrediction
    .from_pretrained(
        HEAD_PROBED_TTM_DIR
    )
    .to(DEVICE)
)

for model in (
    zero_shot_ttm,
    hp_ttm,
):
    for parameter in (
        model.parameters()
    ):
        parameter.requires_grad_(
            False
        )

    model.eval()

assert sum(
    parameter.numel()
    for parameter in zero_shot_ttm.parameters()
    if parameter.requires_grad
) == 0

assert sum(
    parameter.numel()
    for parameter in hp_ttm.parameters()
    if parameter.requires_grad
) == 0

Loading weights:   0%|          | 0/139 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/139 [00:00<?, ?it/s]

In [9]:
zero_shot_forecast = np.empty(
    (
        SERIES_COUNT,
        FORECAST_HORIZON,
    ),
    dtype=np.float32,
)

hp_forecast = np.empty_like(
    zero_shot_forecast
)

hp_decoder_hidden = np.empty(
    (
        SERIES_COUNT,
        1,
        9,
        128,
    ),
    dtype=np.float32,
)

ttm_inference_start = (
    time.perf_counter()
)

with torch.no_grad():
    for start in range(
        0,
        SERIES_COUNT,
        INFERENCE_BATCH_SIZE,
    ):
        end = min(
            start
            + INFERENCE_BATCH_SIZE,
            SERIES_COUNT,
        )

        batch_values = torch.from_numpy(
            test_context[
                start:end
            ]
        ).unsqueeze(
            -1
        ).to(
            DEVICE
        )

        batch_token = torch.full(
            (
                end - start,
            ),
            FREQUENCY_TOKEN,
            dtype=torch.long,
            device=DEVICE,
        )

        zero_output = (
            zero_shot_ttm(
                past_values=batch_values,
                freq_token=batch_token,
                return_dict=True,
            )
        )

        hp_output = (
            hp_ttm(
                past_values=batch_values,
                freq_token=batch_token,
                output_hidden_states=True,
                return_dict=True,
            )
        )

        zero_shot_forecast[
            start:end
        ] = (
            zero_output
            .prediction_outputs
            .squeeze(-1)
            .float()
            .cpu()
            .numpy()
        )

        hp_forecast[
            start:end
        ] = (
            hp_output
            .prediction_outputs
            .squeeze(-1)
            .float()
            .cpu()
            .numpy()
        )

        hp_decoder_hidden[
            start:end
        ] = (
            hp_output
            .decoder_hidden_state
            .float()
            .cpu()
            .numpy()
        )

        del (
            batch_values,
            batch_token,
            zero_output,
            hp_output,
        )

torch.cuda.synchronize()

ttm_inference_runtime_seconds = (
    time.perf_counter()
    - ttm_inference_start
)

assert zero_shot_forecast.shape == (
    30_490,
    28,
)

assert hp_forecast.shape == (
    30_490,
    28,
)

assert hp_decoder_hidden.shape == (
    30_490,
    1,
    9,
    128,
)

## Reconstruct the locked 128-dimensional horizon decoder stream

In [10]:
alignment = np.load(
    METADATA_DIR
    / "05_DECODER_ALIGNMENT_WEIGHTS.npz"
)

selected_weight = alignment[
    "selected_weight"
].astype(
    np.float32,
    copy=False,
)

assert selected_weight.shape == (
    28,
    9,
    128,
)

horizon_decoder_features = np.einsum(
    "bpk,hpk->bhk",
    hp_decoder_hidden[
        :,
        0,
        :,
        :,
    ],
    selected_weight,
    optimize=True,
).astype(
    np.float32,
    copy=False,
)

assert horizon_decoder_features.shape == (
    30_490,
    28,
    128,
)

## Test recent-demand features

The exact Phase 06 formulas and stored training-only scaling statistics are reused.

In [11]:
RECENT_LOOKBACK = 28

recent_x = np.arange(
    RECENT_LOOKBACK,
    dtype=np.float64,
)

recent_x_centered = (
    recent_x
    - recent_x.mean()
)

trend_denominator = float(
    np.square(
        recent_x_centered
    ).sum()
)

recent_window = test_context[
    :,
    -RECENT_LOOKBACK:,
].astype(
    np.float64,
    copy=False,
)

recent_median = np.median(
    recent_window,
    axis=1,
)

recent_mad = np.median(
    np.abs(
        recent_window
        - recent_median[
            :,
            None,
        ]
    ),
    axis=1,
)

recent_scale = (
    1.4826
    * recent_mad
)

recent_change = (
    recent_window[
        :,
        -7:,
    ].mean(
        axis=1
    )
    - recent_window[
        :,
        -14:-7,
    ].mean(
        axis=1
    )
)

recent_centered = (
    recent_window
    - recent_window.mean(
        axis=1,
        keepdims=True,
    )
)

recent_trend = (
    (
        recent_centered
        * recent_x_centered[
            None,
            :,
        ]
    ).sum(
        axis=1
    )
    / trend_denominator
)

recent_zero_rate = (
    recent_window
    == 0.0
).mean(
    axis=1
)

test_recent_raw = np.column_stack(
    [
        recent_median,
        recent_scale,
        recent_change,
        recent_trend,
        recent_zero_rate,
    ]
).astype(
    np.float32
)

test_recent_stabilized = np.empty_like(
    test_recent_raw,
    dtype=np.float32,
)

test_recent_stabilized[
    :,
    0,
] = np.log1p(
    test_recent_raw[
        :,
        0,
    ]
)

test_recent_stabilized[
    :,
    1,
] = np.log1p(
    test_recent_raw[
        :,
        1,
    ]
)

test_recent_stabilized[
    :,
    2,
] = (
    np.sign(
        test_recent_raw[
            :,
            2,
        ]
    )
    * np.log1p(
        np.abs(
            test_recent_raw[
                :,
                2,
            ]
        )
    )
)

test_recent_stabilized[
    :,
    3,
] = (
    np.sign(
        test_recent_raw[
            :,
            3,
        ]
    )
    * np.log1p(
        np.abs(
            test_recent_raw[
                :,
                3,
            ]
        )
    )
)

test_recent_stabilized[
    :,
    4,
] = test_recent_raw[
    :,
    4,
]

recent_feature_mean = np.asarray(
    feature_scaler[
        "recent_feature_mean"
    ],
    dtype=np.float64,
)

recent_feature_std = np.asarray(
    feature_scaler[
        "recent_feature_std"
    ],
    dtype=np.float64,
)

test_recent_features = (
    (
        test_recent_stabilized
        - recent_feature_mean
    )
    / recent_feature_std
).astype(
    np.float32
)

assert test_recent_features.shape == (
    30_490,
    5,
)

assert np.isfinite(
    test_recent_features
).all()

## Test future-known covariates

The Phase 06 feature contract is reused exactly:

- state-specific SNAP;
- event indicator;
- current price-record indicator;
- valid weekly price-change indicator;
- standardized log price;
- standardized weekly log-price change;
- day-of-week sine;
- day-of-week cosine.

In [12]:
calendar = pd.read_parquet(
    (
        PROCESSED_M5_DIR
        / "calendar.parquet"
    ),
    columns=[
        "d",
        "date",
        "wm_yr_wk",
        "wday",
        "event_name_1",
        "event_name_2",
        "snap_CA",
        "snap_TX",
        "snap_WI",
    ],
)

calendar = (
    calendar
    .assign(
        day_index=lambda frame: (
            frame["d"]
            .str.removeprefix(
                "d_"
            )
            .astype(
                "int32"
            )
        )
    )
    .sort_values(
        "day_index",
        kind="stable",
    )
)

calendar_by_day = (
    calendar
    .set_index(
        "day_index"
    )
)

store_ids = (
    test_frame[
        "store_id"
    ]
    .astype(str)
    .to_numpy()
)

item_ids = (
    test_frame[
        "item_id"
    ]
    .astype(str)
    .to_numpy()
)

state_ids = (
    test_frame[
        "state_id"
    ]
    .astype(str)
    .to_numpy()
)

state_to_index = {
    "CA": 0,
    "TX": 1,
    "WI": 2,
}

state_indices = np.asarray(
    [
        state_to_index[
            state
        ]
        for state in state_ids
    ],
    dtype=np.int8,
)

calendar_week_order = (
    calendar[
        [
            "wm_yr_wk",
            "date",
        ]
    ]
    .drop_duplicates(
        "wm_yr_wk"
    )
    .sort_values(
        "date",
        kind="stable",
    )[
        "wm_yr_wk"
    ]
    .astype(int)
    .tolist()
)

previous_week_map = {}

for position, week in enumerate(
    calendar_week_order
):
    previous_week_map[
        week
    ] = (
        calendar_week_order[
            position - 1
        ]
        if position > 0
        else None
    )

test_target_days = list(
    range(
        TEST_ORIGIN + 1,
        TEST_ORIGIN
        + FORECAST_HORIZON
        + 1,
    )
)

test_target_weeks = sorted(
    {
        int(
            calendar_by_day.loc[
                day,
                "wm_yr_wk",
            ]
        )
        for day in test_target_days
    }
)

required_price_weeks = set(
    test_target_weeks
)

for week in test_target_weeks:
    previous_week = (
        previous_week_map[
            week
        ]
    )

    if (
        previous_week
        is not None
    ):
        required_price_weeks.add(
            previous_week
        )

required_price_weeks = sorted(
    required_price_weeks
)

price_table = pd.read_parquet(
    (
        PROCESSED_M5_DIR
        / "sell_prices.parquet"
    ),
    columns=[
        "store_id",
        "item_id",
        "wm_yr_wk",
        "sell_price",
    ],
)

price_table = price_table.loc[
    price_table[
        "wm_yr_wk"
    ].isin(
        required_price_weeks
    )
].copy()

series_price_index = (
    pd.MultiIndex
    .from_arrays(
        [
            store_ids,
            item_ids,
        ],
        names=[
            "store_id",
            "item_id",
        ],
    )
)

price_pivot = (
    price_table
    .pivot(
        index=[
            "store_id",
            "item_id",
        ],
        columns="wm_yr_wk",
        values="sell_price",
    )
    .reindex(
        series_price_index
    )
)

for week in (
    required_price_weeks
):
    if (
        week
        not in price_pivot.columns
    ):
        price_pivot[
            week
        ] = np.nan

price_pivot = price_pivot[
    required_price_weeks
]

price_matrix = price_pivot.to_numpy(
    dtype=np.float32,
    copy=True,
)

week_to_column_index = {
    int(week): index
    for index, week in enumerate(
        required_price_weeks
    )
}

In [13]:
test_future_raw = np.empty(
    (
        SERIES_COUNT,
        FORECAST_HORIZON,
        8,
    ),
    dtype=np.float32,
)

for horizon_index, target_day in enumerate(
    test_target_days
):
    calendar_row = (
        calendar_by_day.loc[
            target_day
        ]
    )

    snap_by_state = np.asarray(
        [
            calendar_row[
                "snap_CA"
            ],
            calendar_row[
                "snap_TX"
            ],
            calendar_row[
                "snap_WI"
            ],
        ],
        dtype=np.float32,
    )

    snap_indicator = (
        snap_by_state[
            state_indices
        ]
    )

    event_indicator = np.float32(
        pd.notna(
            calendar_row[
                "event_name_1"
            ]
        )
        or pd.notna(
            calendar_row[
                "event_name_2"
            ]
        )
    )

    current_week = int(
        calendar_row[
            "wm_yr_wk"
        ]
    )

    previous_week = (
        previous_week_map[
            current_week
        ]
    )

    current_price = (
        price_matrix[
            :,
            week_to_column_index[
                current_week
            ],
        ]
    )

    current_observed = (
        np.isfinite(
            current_price
        )
        & (
            current_price
            > 0.0
        )
    )

    if (
        previous_week
        is not None
        and previous_week
        in week_to_column_index
    ):
        previous_price = (
            price_matrix[
                :,
                week_to_column_index[
                    previous_week
                ],
            ]
        )

        previous_observed = (
            np.isfinite(
                previous_price
            )
            & (
                previous_price
                > 0.0
            )
        )
    else:
        previous_price = np.full(
            SERIES_COUNT,
            np.nan,
            dtype=np.float32,
        )

        previous_observed = np.zeros(
            SERIES_COUNT,
            dtype=bool,
        )

    price_change_valid = (
        current_observed
        & previous_observed
    )

    log_price = np.zeros(
        SERIES_COUNT,
        dtype=np.float32,
    )

    log_price[
        current_observed
    ] = np.log1p(
        current_price[
            current_observed
        ]
    ).astype(
        np.float32
    )

    log_price_change = np.zeros(
        SERIES_COUNT,
        dtype=np.float32,
    )

    log_price_change[
        price_change_valid
    ] = np.log(
        current_price[
            price_change_valid
        ]
        / previous_price[
            price_change_valid
        ]
    ).astype(
        np.float32
    )

    wday = int(
        calendar_row[
            "wday"
        ]
    )

    angle = (
        2.0
        * math.pi
        * (wday - 1)
        / 7.0
    )

    test_future_raw[
        :,
        horizon_index,
        0,
    ] = snap_indicator

    test_future_raw[
        :,
        horizon_index,
        1,
    ] = event_indicator

    test_future_raw[
        :,
        horizon_index,
        2,
    ] = current_observed.astype(
        np.float32
    )

    test_future_raw[
        :,
        horizon_index,
        3,
    ] = price_change_valid.astype(
        np.float32
    )

    test_future_raw[
        :,
        horizon_index,
        4,
    ] = log_price

    test_future_raw[
        :,
        horizon_index,
        5,
    ] = log_price_change

    test_future_raw[
        :,
        horizon_index,
        6,
    ] = np.float32(
        math.sin(
            angle
        )
    )

    test_future_raw[
        :,
        horizon_index,
        7,
    ] = np.float32(
        math.cos(
            angle
        )
    )

In [14]:
price_scaler = (
    feature_scaler[
        "continuous_future_scaling"
    ]
)

price_mean = float(
    price_scaler[
        "log_price"
    ]["mean"]
)

price_std = float(
    price_scaler[
        "log_price"
    ]["std"]
)

price_change_mean = float(
    price_scaler[
        "weekly_log_price_change"
    ]["mean"]
)

price_change_std = float(
    price_scaler[
        "weekly_log_price_change"
    ]["std"]
)

test_future_covariates = (
    test_future_raw.copy()
)

price_observed_mask = (
    test_future_raw[
        :,
        :,
        2,
    ]
    == 1.0
)

price_change_valid_mask = (
    test_future_raw[
        :,
        :,
        3,
    ]
    == 1.0
)

raw_price = (
    test_future_raw[
        :,
        :,
        4,
    ].copy()
)

raw_change = (
    test_future_raw[
        :,
        :,
        5,
    ].copy()
)

test_future_covariates[
    :,
    :,
    4,
] = 0.0

test_future_covariates[
    :,
    :,
    5,
] = 0.0

test_future_covariates[
    :,
    :,
    4,
][
    price_observed_mask
] = (
    (
        raw_price[
            price_observed_mask
        ]
        - price_mean
    )
    / price_std
)

test_future_covariates[
    :,
    :,
    5,
][
    price_change_valid_mask
] = (
    (
        raw_change[
            price_change_valid_mask
        ]
        - price_change_mean
    )
    / price_change_std
)

test_future_covariates = (
    test_future_covariates
    .astype(
        np.float32,
        copy=False,
    )
)

assert test_future_covariates.shape == (
    30_490,
    28,
    8,
)

assert np.isfinite(
    test_future_covariates
).all()

## Locked test extreme labels

The Phase 07 threshold is reused without modification.

In [15]:
ROBUST_SCALE_FLOOR = float(
    extreme_contract[
        "robust_scale_floor"
    ]
)

EXTREME_THRESHOLD = float(
    extreme_contract[
        "extreme_threshold"
    ]
)

test_effective_scale = np.maximum(
    recent_scale,
    ROBUST_SCALE_FLOOR,
).astype(
    np.float32
)

test_shock_score = (
    np.abs(
        test_target
        - recent_median[
            :,
            None,
        ]
    )
    / test_effective_scale[
        :,
        None,
    ]
).astype(
    np.float32
)

test_extreme_label = (
    test_shock_score
    > EXTREME_THRESHOLD
)

test_signed_deviation = (
    test_target
    - recent_median[
        :,
        None,
    ]
)

test_positive_extreme = (
    test_extreme_label
    & (
        test_signed_deviation
        > 0.0
    )
)

test_negative_extreme = (
    test_extreme_label
    & (
        test_signed_deviation
        < 0.0
    )
)

test_normal = (
    ~test_extreme_label
)

test_extreme_rate = float(
    test_extreme_label.mean(
        dtype=np.float64
    )
)

test_positive_extreme_rate = float(
    test_positive_extreme.mean(
        dtype=np.float64
    )
)

test_negative_extreme_rate = float(
    test_negative_extreme.mean(
        dtype=np.float64
    )
)

print(
    "Locked-test extreme rate:",
    test_extreme_rate,
)

print(
    "Positive extreme rate:",
    test_positive_extreme_rate,
)

print(
    "Negative extreme rate:",
    test_negative_extreme_rate,
)

Locked-test extreme rate: 0.05927470364990863
Positive extreme rate: 0.05546432085461275
Negative extreme rate: 0.0038103827952958814


## Fixed adapter architectures

In [16]:
class GatedAdapter(nn.Module):
    def __init__(
        self,
        input_dim: int,
        extreme_prior: float,
        encoder_width: int = 16,
        correction_hidden_width: int = 8,
    ) -> None:
        super().__init__()

        self.shock_encoder = nn.Sequential(
            nn.Linear(
                input_dim,
                encoder_width,
            ),
            nn.GELU(),
            nn.LayerNorm(
                encoder_width,
            ),
        )

        self.extreme_gate = nn.Linear(
            encoder_width,
            1,
        )

        self.correction_head = nn.Sequential(
            nn.Linear(
                encoder_width,
                correction_hidden_width,
            ),
            nn.GELU(),
            nn.Linear(
                correction_hidden_width,
                1,
            ),
        )

        nn.init.zeros_(
            self.extreme_gate.weight
        )

        prior_logit = math.log(
            extreme_prior
            / (
                1.0
                - extreme_prior
            )
        )

        nn.init.constant_(
            self.extreme_gate.bias,
            prior_logit,
        )

        nn.init.zeros_(
            self.correction_head[
                -1
            ].weight
        )

        nn.init.zeros_(
            self.correction_head[
                -1
            ].bias
        )

    def forward(
        self,
        input_tensor: torch.Tensor,
        base_forecast: torch.Tensor,
    ) -> dict[str, torch.Tensor]:
        representation = (
            self.shock_encoder(
                input_tensor
            )
        )

        gate_logits = (
            self.extreme_gate(
                representation
            ).squeeze(
                -1
            )
        )

        gate_probability = (
            torch.sigmoid(
                gate_logits
            )
        )

        residual_correction = (
            self.correction_head(
                representation
            ).squeeze(
                -1
            )
        )

        final_forecast = (
            base_forecast
            + gate_probability
            * residual_correction
        )

        return {
            "gate_probability": gate_probability,
            "residual_correction": residual_correction,
            "final_forecast": final_forecast,
        }


class UngatedCrossfeedAdapter(nn.Module):
    def __init__(
        self,
        input_dim: int = 141,
        encoder_width: int = 16,
        correction_hidden_width: int = 8,
    ) -> None:
        super().__init__()

        self.shock_encoder = nn.Sequential(
            nn.Linear(
                input_dim,
                encoder_width,
            ),
            nn.GELU(),
            nn.LayerNorm(
                encoder_width,
            ),
        )

        self.correction_head = nn.Sequential(
            nn.Linear(
                encoder_width,
                correction_hidden_width,
            ),
            nn.GELU(),
            nn.Linear(
                correction_hidden_width,
                1,
            ),
        )

        nn.init.zeros_(
            self.correction_head[
                -1
            ].weight
        )

        nn.init.zeros_(
            self.correction_head[
                -1
            ].bias
        )

    def forward(
        self,
        input_tensor: torch.Tensor,
        base_forecast: torch.Tensor,
    ) -> dict[str, torch.Tensor]:
        representation = (
            self.shock_encoder(
                input_tensor
            )
        )

        residual_correction = (
            self.correction_head(
                representation
            ).squeeze(
                -1
            )
        )

        final_forecast = (
            base_forecast
            + residual_correction
        )

        return {
            "residual_correction": residual_correction,
            "final_forecast": final_forecast,
        }

## Load fixed adapter checkpoints

In [17]:
training_extreme_prior = float(
    extreme_contract[
        "training_extreme_rate"
    ]
)

primary_adapter = GatedAdapter(
    input_dim=141,
    extreme_prior=training_extreme_prior,
).to(DEVICE)

primary_adapter.load_state_dict(
    torch.load(
        PRIMARY_ADAPTER_DIR
        / "adapter_state.pt",
        map_location=DEVICE,
        weights_only=True,
    )
)

retail_only_adapter = GatedAdapter(
    input_dim=13,
    extreme_prior=training_extreme_prior,
).to(DEVICE)

retail_only_checkpoint = torch.load(
    PHASE09_CHECKPOINT_DIR
    / "retail_only_gated_best_state.pt",
    map_location=DEVICE,
    weights_only=False,
)

retail_only_adapter.load_state_dict(
    retail_only_checkpoint[
        "model_state_dict"
    ]
)

ungated_adapter = (
    UngatedCrossfeedAdapter(
        input_dim=141,
    ).to(DEVICE)
)

ungated_checkpoint = torch.load(
    PHASE09_CHECKPOINT_DIR
    / "ungated_crossfeed_best_state.pt",
    map_location=DEVICE,
    weights_only=False,
)

ungated_adapter.load_state_dict(
    ungated_checkpoint[
        "model_state_dict"
    ]
)

for model in (
    primary_adapter,
    retail_only_adapter,
    ungated_adapter,
):
    for parameter in (
        model.parameters()
    ):
        parameter.requires_grad_(
            False
        )

    model.eval()

## Fixed adapter inference

In [18]:
primary_ea_forecast = np.empty(
    (
        SERIES_COUNT,
        FORECAST_HORIZON,
    ),
    dtype=np.float32,
)

primary_gate_probability = np.empty_like(
    primary_ea_forecast
)

primary_residual_correction = np.empty_like(
    primary_ea_forecast
)

retail_only_forecast = np.empty_like(
    primary_ea_forecast
)

retail_only_gate_probability = np.empty_like(
    primary_ea_forecast
)

ungated_forecast = np.empty_like(
    primary_ea_forecast
)

adapter_inference_start = (
    time.perf_counter()
)

with torch.no_grad():
    for start in range(
        0,
        SERIES_COUNT,
        INFERENCE_BATCH_SIZE,
    ):
        end = min(
            start
            + INFERENCE_BATCH_SIZE,
            SERIES_COUNT,
        )

        recent = torch.from_numpy(
            test_recent_features[
                start:end
            ]
        ).to(
            DEVICE
        )

        decoder = torch.from_numpy(
            horizon_decoder_features[
                start:end
            ]
        ).to(
            DEVICE
        )

        future = torch.from_numpy(
            test_future_covariates[
                start:end
            ]
        ).to(
            DEVICE
        )

        base = torch.from_numpy(
            hp_forecast[
                start:end
            ]
        ).to(
            DEVICE
        )

        recent_expanded = (
            recent
            .unsqueeze(1)
            .expand(
                -1,
                FORECAST_HORIZON,
                -1,
            )
        )

        full_input = torch.cat(
            [
                recent_expanded,
                decoder,
                future,
            ],
            dim=-1,
        )

        retail_input = torch.cat(
            [
                recent_expanded,
                future,
            ],
            dim=-1,
        )

        primary_output = (
            primary_adapter(
                input_tensor=full_input,
                base_forecast=base,
            )
        )

        retail_output = (
            retail_only_adapter(
                input_tensor=retail_input,
                base_forecast=base,
            )
        )

        ungated_output = (
            ungated_adapter(
                input_tensor=full_input,
                base_forecast=base,
            )
        )

        primary_ea_forecast[
            start:end
        ] = (
            primary_output[
                "final_forecast"
            ]
            .cpu()
            .numpy()
        )

        primary_gate_probability[
            start:end
        ] = (
            primary_output[
                "gate_probability"
            ]
            .cpu()
            .numpy()
        )

        primary_residual_correction[
            start:end
        ] = (
            primary_output[
                "residual_correction"
            ]
            .cpu()
            .numpy()
        )

        retail_only_forecast[
            start:end
        ] = (
            retail_output[
                "final_forecast"
            ]
            .cpu()
            .numpy()
        )

        retail_only_gate_probability[
            start:end
        ] = (
            retail_output[
                "gate_probability"
            ]
            .cpu()
            .numpy()
        )

        ungated_forecast[
            start:end
        ] = (
            ungated_output[
                "final_forecast"
            ]
            .cpu()
            .numpy()
        )

        del (
            recent,
            decoder,
            future,
            base,
            recent_expanded,
            full_input,
            retail_input,
            primary_output,
            retail_output,
            ungated_output,
        )

torch.cuda.synchronize()

adapter_inference_runtime_seconds = (
    time.perf_counter()
    - adapter_inference_start
)

## Final-test metric functions

In [19]:
def forecasting_metrics(
    actual: np.ndarray,
    predicted: np.ndarray,
) -> dict[str, float]:
    actual_64 = actual.astype(
        np.float64,
        copy=False,
    )

    predicted_64 = predicted.astype(
        np.float64,
        copy=False,
    )

    error = (
        predicted_64
        - actual_64
    )

    absolute_error = np.abs(
        error
    )

    squared_error = np.square(
        error
    )

    mse = float(
        squared_error.mean()
    )

    rmse = float(
        np.sqrt(
            mse
        )
    )

    mae = float(
        absolute_error.mean()
    )

    denominator = float(
        np.abs(
            actual_64
        ).sum()
    )

    wape = float(
        absolute_error.sum()
        / denominator
    )

    return {
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "wape": wape,
    }

def masked_mse(
    actual: np.ndarray,
    predicted: np.ndarray,
    mask: np.ndarray,
) -> float:
    assert mask.any()

    error = (
        predicted.astype(
            np.float64,
            copy=False,
        )
        - actual.astype(
            np.float64,
            copy=False,
        )
    )

    return float(
        np.square(
            error[
                mask
            ]
        ).mean()
    )

def improvement_percent(
    baseline: float,
    candidate: float,
) -> float:
    return float(
        100.0
        * (
            baseline
            - candidate
        )
        / baseline
    )

## Final model comparison

In [20]:
model_forecasts = {
    "Weekly Seasonal Naive": (
        seasonal_naive_forecast
    ),
    "Zero-shot TTM": (
        zero_shot_forecast
    ),
    "Frozen HP-TTM": (
        hp_forecast
    ),
    "Retail-only gated adapter": (
        retail_only_forecast
    ),
    "Ungated cross-feed adapter": (
        ungated_forecast
    ),
    "Primary EA-TTM": (
        primary_ea_forecast
    ),
}

overall_metric_records = []
regime_metric_records = []

for model_name, forecast in (
    model_forecasts.items()
):
    overall_metrics = (
        forecasting_metrics(
            test_target,
            forecast,
        )
    )

    overall_metric_records.append(
        {
            "model": model_name,
            **overall_metrics,
        }
    )

    regime_metric_records.append(
        {
            "model": model_name,
            "normal_mse": masked_mse(
                test_target,
                forecast,
                test_normal,
            ),
            "extreme_mse": masked_mse(
                test_target,
                forecast,
                test_extreme_label,
            ),
            "positive_extreme_mse": masked_mse(
                test_target,
                forecast,
                test_positive_extreme,
            ),
            "negative_extreme_mse": masked_mse(
                test_target,
                forecast,
                test_negative_extreme,
            ),
        }
    )

overall_metrics_table = pd.DataFrame(
    overall_metric_records
)

regime_metrics_table = pd.DataFrame(
    regime_metric_records
)

overall_metrics_table

,model,mse,rmse,mae,wape
0,Weekly Seasonal Naive,7.165766,2.676895,1.243972,0.862181
1,Zero-shot TTM,4.637857,2.153569,1.041414,0.721791
2,Frozen HP-TTM,4.563047,2.136129,1.028902,0.713119
3,Retail-only gated adapter,4.532796,2.129037,1.038483,0.719759
4,Ungated cross-feed adapter,4.513243,2.124439,1.041909,0.722134
5,Primary EA-TTM,4.528610,2.128053,1.037919,0.719369


## Primary EA-TTM final mechanism evidence

In [21]:
primary_effective_correction = (
    primary_gate_probability
    * primary_residual_correction
)

primary_gate_auroc = float(
    roc_auc_score(
        test_extreme_label.ravel(),
        primary_gate_probability.ravel(),
    )
)

primary_gate_auprc = float(
    average_precision_score(
        test_extreme_label.ravel(),
        primary_gate_probability.ravel(),
    )
)

primary_gate_auprc_lift = float(
    primary_gate_auprc
    / test_extreme_rate
)

mean_gate_extreme = float(
    primary_gate_probability[
        test_extreme_label
    ].mean(
        dtype=np.float64
    )
)

mean_gate_normal = float(
    primary_gate_probability[
        test_normal
    ].mean(
        dtype=np.float64
    )
)

mean_abs_effective_correction_extreme = float(
    np.abs(
        primary_effective_correction[
            test_extreme_label
        ]
    ).mean(
        dtype=np.float64
    )
)

mean_abs_effective_correction_normal = float(
    np.abs(
        primary_effective_correction[
            test_normal
        ]
    ).mean(
        dtype=np.float64
    )
)

effective_correction_ratio = float(
    mean_abs_effective_correction_extreme
    / mean_abs_effective_correction_normal
)

hp_squared_error = np.square(
    hp_forecast.astype(
        np.float64
    )
    - test_target.astype(
        np.float64
    )
)

ea_squared_error = np.square(
    primary_ea_forecast.astype(
        np.float64
    )
    - test_target.astype(
        np.float64
    )
)

extreme_point_improvement_rate = float(
    (
        ea_squared_error[
            test_extreme_label
        ]
        < hp_squared_error[
            test_extreme_label
        ]
    ).mean()
)

normal_point_improvement_rate = float(
    (
        ea_squared_error[
            test_normal
        ]
        < hp_squared_error[
            test_normal
        ]
    ).mean()
)

true_residual = (
    test_target
    - hp_forecast
)

nonzero_residual_mask = (
    np.abs(
        true_residual
    )
    > 1e-12
)

overall_direction_agreement = float(
    (
        np.sign(
            primary_effective_correction[
                nonzero_residual_mask
            ]
        )
        == np.sign(
            true_residual[
                nonzero_residual_mask
            ]
        )
    ).mean()
)

extreme_direction_mask = (
    test_extreme_label
    & nonzero_residual_mask
)

extreme_direction_agreement = float(
    (
        np.sign(
            primary_effective_correction[
                extreme_direction_mask
            ]
        )
        == np.sign(
            true_residual[
                extreme_direction_mask
            ]
        )
    ).mean()
)

## Paired series bootstrap: Primary EA-TTM versus HP-TTM

In [22]:
BOOTSTRAP_REPLICATES = 2_000

hp_series_mse = np.square(
    hp_forecast.astype(
        np.float64
    )
    - test_target.astype(
        np.float64
    )
).mean(
    axis=1
)

ea_series_mse = np.square(
    primary_ea_forecast.astype(
        np.float64
    )
    - test_target.astype(
        np.float64
    )
).mean(
    axis=1
)

paired_series_difference = (
    hp_series_mse
    - ea_series_mse
)

observed_paired_mse_difference = float(
    paired_series_difference.mean()
)

bootstrap_rng = np.random.default_rng(
    BASE_SEED
)

bootstrap_differences = np.empty(
    BOOTSTRAP_REPLICATES,
    dtype=np.float64,
)

for replicate in range(
    BOOTSTRAP_REPLICATES
):
    sampled_indices = (
        bootstrap_rng.integers(
            0,
            SERIES_COUNT,
            size=SERIES_COUNT,
        )
    )

    bootstrap_differences[
        replicate
    ] = (
        paired_series_difference[
            sampled_indices
        ].mean()
    )

bootstrap_ci_lower = float(
    np.quantile(
        bootstrap_differences,
        0.025,
    )
)

bootstrap_ci_upper = float(
    np.quantile(
        bootstrap_differences,
        0.975,
    )
)

bootstrap_probability_positive = float(
    (
        bootstrap_differences
        > 0.0
    ).mean()
)

## Final comparison statistics

In [23]:
overall_metrics_by_model = (
    overall_metrics_table
    .set_index(
        "model"
    )
    .to_dict(
        orient="index"
    )
)

regime_metrics_by_model = (
    regime_metrics_table
    .set_index(
        "model"
    )
    .to_dict(
        orient="index"
    )
)

hp_overall = (
    overall_metrics_by_model[
        "Frozen HP-TTM"
    ]
)

primary_overall = (
    overall_metrics_by_model[
        "Primary EA-TTM"
    ]
)

hp_regime = (
    regime_metrics_by_model[
        "Frozen HP-TTM"
    ]
)

primary_regime = (
    regime_metrics_by_model[
        "Primary EA-TTM"
    ]
)

primary_vs_hp = {
    "overall_mse_improvement_percent": (
        improvement_percent(
            hp_overall["mse"],
            primary_overall["mse"],
        )
    ),
    "overall_rmse_improvement_percent": (
        improvement_percent(
            hp_overall["rmse"],
            primary_overall["rmse"],
        )
    ),
    "overall_mae_improvement_percent": (
        improvement_percent(
            hp_overall["mae"],
            primary_overall["mae"],
        )
    ),
    "overall_wape_improvement_percent": (
        improvement_percent(
            hp_overall["wape"],
            primary_overall["wape"],
        )
    ),
    "normal_mse_improvement_percent": (
        improvement_percent(
            hp_regime["normal_mse"],
            primary_regime["normal_mse"],
        )
    ),
    "extreme_mse_improvement_percent": (
        improvement_percent(
            hp_regime["extreme_mse"],
            primary_regime["extreme_mse"],
        )
    ),
    "positive_extreme_mse_improvement_percent": (
        improvement_percent(
            hp_regime[
                "positive_extreme_mse"
            ],
            primary_regime[
                "positive_extreme_mse"
            ],
        )
    ),
    "negative_extreme_mse_improvement_percent": (
        improvement_percent(
            hp_regime[
                "negative_extreme_mse"
            ],
            primary_regime[
                "negative_extreme_mse"
            ],
        )
    ),
}

ungated_vs_primary = {
    "overall_mse_percent": (
        improvement_percent(
            primary_overall["mse"],
            overall_metrics_by_model[
                "Ungated cross-feed adapter"
            ]["mse"],
        )
    ),
    "extreme_mse_percent": (
        improvement_percent(
            primary_regime["extreme_mse"],
            regime_metrics_by_model[
                "Ungated cross-feed adapter"
            ]["extreme_mse"],
        )
    ),
}

## Persist final-test evidence and seal the project

In [24]:
FINAL_METRICS_PATH = (
    RESULTS_TABLE_DIR
    / "10_FINAL_TEST_METRICS.csv"
)

REGIME_METRICS_PATH = (
    RESULTS_TABLE_DIR
    / "10_FINAL_TEST_REGIME_METRICS.csv"
)

MECHANISM_PATH = (
    RESULTS_TABLE_DIR
    / "10_FINAL_TEST_MECHANISM.csv"
)

BOOTSTRAP_PATH = (
    RESULTS_TABLE_DIR
    / "10_FINAL_TEST_BOOTSTRAP.csv"
)

PREDICTIONS_PATH = (
    RUN_DIR
    / "10_FINAL_TEST_OUTPUTS.npz"
)

FINAL_EVALUATION_LOCK_PATH = (
    METADATA_DIR
    / "10_FINAL_EVALUATION_LOCK.json"
)

RUN_SUMMARY_PATH = (
    METADATA_DIR
    / "10_RUN_SUMMARY.json"
)

overall_metrics_table.to_csv(
    FINAL_METRICS_PATH,
    index=False,
)

regime_metrics_table.to_csv(
    REGIME_METRICS_PATH,
    index=False,
)

mechanism_evidence = {
    "test_extreme_rate": test_extreme_rate,
    "test_positive_extreme_rate": test_positive_extreme_rate,
    "test_negative_extreme_rate": test_negative_extreme_rate,
    "gate_auroc": primary_gate_auroc,
    "gate_auprc": primary_gate_auprc,
    "gate_auprc_lift_vs_prevalence": primary_gate_auprc_lift,
    "mean_gate_extreme": mean_gate_extreme,
    "mean_gate_normal": mean_gate_normal,
    "mean_abs_effective_correction_extreme": mean_abs_effective_correction_extreme,
    "mean_abs_effective_correction_normal": mean_abs_effective_correction_normal,
    "effective_correction_extreme_to_normal_ratio": effective_correction_ratio,
    "extreme_point_squared_error_improvement_rate": extreme_point_improvement_rate,
    "normal_point_squared_error_improvement_rate": normal_point_improvement_rate,
    "overall_correction_direction_agreement": overall_direction_agreement,
    "extreme_correction_direction_agreement": extreme_direction_agreement,
}

pd.DataFrame(
    [
        {
            "metric": key,
            "value": value,
        }
        for key, value in (
            mechanism_evidence.items()
        )
    ]
).to_csv(
    MECHANISM_PATH,
    index=False,
)

bootstrap_table = pd.DataFrame(
    [
        {
            "comparison": "Primary EA-TTM vs Frozen HP-TTM",
            "statistic": "paired series MSE difference (HP - EA)",
            "observed_difference": observed_paired_mse_difference,
            "bootstrap_replicates": BOOTSTRAP_REPLICATES,
            "ci_95_lower": bootstrap_ci_lower,
            "ci_95_upper": bootstrap_ci_upper,
            "bootstrap_probability_difference_positive": bootstrap_probability_positive,
        }
    ]
)

bootstrap_table.to_csv(
    BOOTSTRAP_PATH,
    index=False,
)

np.savez_compressed(
    PREDICTIONS_PATH,
    actual=test_target.astype(
        np.float32
    ),
    seasonal_naive=seasonal_naive_forecast.astype(
        np.float32
    ),
    zero_shot_ttm=zero_shot_forecast.astype(
        np.float32
    ),
    hp_ttm=hp_forecast.astype(
        np.float32
    ),
    retail_only_gated=retail_only_forecast.astype(
        np.float32
    ),
    ungated_crossfeed=ungated_forecast.astype(
        np.float32
    ),
    primary_ea_ttm=primary_ea_forecast.astype(
        np.float32
    ),
    primary_gate_probability=primary_gate_probability.astype(
        np.float32
    ),
    primary_residual_correction=primary_residual_correction.astype(
        np.float32
    ),
    extreme_label=test_extreme_label.astype(
        np.uint8
    ),
    positive_extreme=test_positive_extreme.astype(
        np.uint8
    ),
    negative_extreme=test_negative_extreme.astype(
        np.uint8
    ),
)

In [25]:
final_evaluation_lock = {
    "status": "FINAL_TEST_EVALUATION_COMPLETE",
    "dataset_sha256": DATASET_SHA256,
    "split_sha256": SPLIT_SHA256,
    "development_lock_sha256": DEVELOPMENT_LOCK_SHA256,
    "primary_model": "Phase 08 best-balanced EA-TTM",
    "primary_epoch": 15,
    "final_test_interval": [
        "d_1914",
        "d_1941",
    ],
    "models_evaluated": list(
        model_forecasts.keys()
    ),
    "post_test_tuning_permitted": False,
    "test_results_may_change_model_selection": False,
    "bootstrap_replicates": BOOTSTRAP_REPLICATES,
}

lock_payload = json.dumps(
    final_evaluation_lock,
    sort_keys=True,
    separators=(",", ":"),
).encode(
    "utf-8"
)

final_evaluation_lock_sha256 = (
    hashlib.sha256(
        lock_payload
    ).hexdigest()
)

final_evaluation_lock[
    "final_evaluation_lock_sha256"
] = final_evaluation_lock_sha256

FINAL_EVALUATION_LOCK_PATH.write_text(
    json.dumps(
        final_evaluation_lock,
        indent=2,
    ),
    encoding="utf-8",
)

866

In [26]:
run_summary = {
    "status": "PHASE_10_COMPLETE",
    "dataset_sha256": DATASET_SHA256,
    "split_sha256": SPLIT_SHA256,
    "development_lock_sha256": DEVELOPMENT_LOCK_SHA256,
    "final_evaluation_lock_sha256": final_evaluation_lock_sha256,
    "final_test_interval": [
        "d_1914",
        "d_1941",
    ],
    "test_extreme_rates": {
        "all": test_extreme_rate,
        "positive": test_positive_extreme_rate,
        "negative": test_negative_extreme_rate,
    },
    "overall_metrics": overall_metrics_by_model,
    "regime_metrics": regime_metrics_by_model,
    "primary_ea_vs_hp_percent": primary_vs_hp,
    "ungated_vs_primary_percent": ungated_vs_primary,
    "primary_mechanism": mechanism_evidence,
    "paired_bootstrap": {
        "observed_mse_difference_hp_minus_ea": observed_paired_mse_difference,
        "ci_95_lower": bootstrap_ci_lower,
        "ci_95_upper": bootstrap_ci_upper,
        "probability_difference_positive": bootstrap_probability_positive,
        "replicates": BOOTSTRAP_REPLICATES,
    },
    "runtime_seconds": {
        "ttm_inference": ttm_inference_runtime_seconds,
        "adapter_inference": adapter_inference_runtime_seconds,
    },
    "post_test_tuning_permitted": False,
}

RUN_SUMMARY_PATH.write_text(
    json.dumps(
        run_summary,
        indent=2,
    ),
    encoding="utf-8",
)

4880

In [27]:
status_path = (
    PROJECT_ROOT
    / "PROJECT_STATUS.md"
)

status_lines = [
    "# EA-TTM Project Status",
    "",
    "## Current stage",
    "Phase 10 — Locked final-test evaluation completed.",
    "",
    "## Primary model",
    "- Phase 08 best-balanced EA-TTM",
    "- Selected epoch: 15",
    "",
    "## Final-test interval",
    "- d_1914 through d_1941",
    "",
    "## Governance",
    "- Model development was locked before final-test access",
    "- Post-test tuning is prohibited",
    "- Final-test results cannot change checkpoint/model selection",
    "",
    f"- Final evaluation lock SHA-256: `{final_evaluation_lock_sha256}`",
    "",
    "## Next stage",
    (
        "Reporting, visualization, manuscript preparation, "
        "and defense evidence only. No additional model tuning."
    ),
]

status_path.write_text(
    "\n".join(
        status_lines
    ),
    encoding="utf-8",
)

598

## Phase validation

Phase 10 passes only if:

$$
\begin{aligned}
&\text{the Phase 09 development lock is verified before test access}
\\
&\land\
\text{the final-test interval is exactly }d_{1914}:d_{1941}
\\
&\land\
\text{all evaluated models were fixed before test access}
\\
&\land\
\text{all predictions are finite}
\\
&\land\
\text{final metrics and predictions are persisted}
\\
&\land\
\text{post-test tuning is prohibited}.
\end{aligned}
$$

In [28]:
all_predictions_finite = all(
    np.isfinite(
        forecast
    ).all()
    for forecast in (
        model_forecasts.values()
    )
)

phase_validation = {
    "development_lock_verified": (
        development_lock[
            "development_lock_sha256"
        ]
        == DEVELOPMENT_LOCK_SHA256
    ),
    "test_interval_valid": (
        target_columns[0] == "d_1914"
        and target_columns[-1] == "d_1941"
    ),
    "primary_model_fixed": (
        development_lock[
            "primary_model"
        ]
        == "Phase 08 best-balanced EA-TTM"
    ),
    "primary_epoch_fixed": (
        int(
            development_lock[
                "primary_epoch"
            ]
        )
        == 15
    ),
    "all_predictions_finite": bool(
        all_predictions_finite
    ),
    "overall_metrics_saved": (
        FINAL_METRICS_PATH.exists()
    ),
    "regime_metrics_saved": (
        REGIME_METRICS_PATH.exists()
    ),
    "predictions_saved": (
        PREDICTIONS_PATH.exists()
    ),
    "final_evaluation_lock_saved": (
        FINAL_EVALUATION_LOCK_PATH.exists()
    ),
    "post_test_tuning_disabled": (
        final_evaluation_lock[
            "post_test_tuning_permitted"
        ]
        is False
    ),
    "run_summary_saved": (
        RUN_SUMMARY_PATH.exists()
    ),
}

assert all(
    phase_validation.values()
), (
    f"Phase 10 validation failed: "
    f"{phase_validation}"
)

phase_validation

{'development_lock_verified': True,
 'test_interval_valid': True,
 'primary_model_fixed': True,
 'primary_epoch_fixed': True,
 'all_predictions_finite': True,
 'overall_metrics_saved': True,
 'regime_metrics_saved': True,
 'predictions_saved': True,
 'final_evaluation_lock_saved': True,
 'post_test_tuning_disabled': True,
 'run_summary_saved': True}

## Return block

The final block is the **only successful-run output that needs to be returned to ChatGPT**.

In [29]:
return_block = {
    "status": "PHASE_10_COMPLETE",
    "dataset_sha256": DATASET_SHA256,
    "split_sha256": SPLIT_SHA256,
    "development_lock_sha256": DEVELOPMENT_LOCK_SHA256,
    "final_evaluation_lock_sha256": final_evaluation_lock_sha256,
    "primary_model": {
        "name": "Phase 08 best-balanced EA-TTM",
        "epoch": 15,
    },
    "final_test_interval": [
        "d_1914",
        "d_1941",
    ],
    "test_extreme_rates": {
        "all": test_extreme_rate,
        "positive": test_positive_extreme_rate,
        "negative": test_negative_extreme_rate,
    },
    "overall_metrics": overall_metrics_by_model,
    "regime_metrics": regime_metrics_by_model,
    "primary_ea_vs_hp_percent": primary_vs_hp,
    "ungated_vs_primary_percent": ungated_vs_primary,
    "primary_mechanism": mechanism_evidence,
    "paired_bootstrap_ea_vs_hp": {
        "observed_mse_difference_hp_minus_ea": observed_paired_mse_difference,
        "ci_95_lower": bootstrap_ci_lower,
        "ci_95_upper": bootstrap_ci_upper,
        "probability_difference_positive": bootstrap_probability_positive,
        "replicates": BOOTSTRAP_REPLICATES,
    },
    "runtime_seconds": {
        "ttm_inference": ttm_inference_runtime_seconds,
        "adapter_inference": adapter_inference_runtime_seconds,
    },
    "post_test_tuning_permitted": False,
    "artifacts": {
        "final_metrics": str(
            FINAL_METRICS_PATH
        ),
        "regime_metrics": str(
            REGIME_METRICS_PATH
        ),
        "mechanism_evidence": str(
            MECHANISM_PATH
        ),
        "bootstrap_evidence": str(
            BOOTSTRAP_PATH
        ),
        "final_test_outputs": str(
            PREDICTIONS_PATH
        ),
        "final_evaluation_lock": str(
            FINAL_EVALUATION_LOCK_PATH
        ),
        "run_summary": str(
            RUN_SUMMARY_PATH
        ),
        "project_status": str(
            status_path
        ),
    },
}

print("PHASE 10 RETURN BLOCK")
print(
    json.dumps(
        return_block,
        indent=2,
    )
)

PHASE 10 RETURN BLOCK
{
  "status": "PHASE_10_COMPLETE",
  "dataset_sha256": "83962ce8bbcb0757215d045f81cac3a7cf4465943334ffa168af326bedc5ed59",
  "split_sha256": "5ff2e30f43a832bb9e513153bb15daa4e1e67c871b43586f2f2f11ef51396447",
  "development_lock_sha256": "b68b706f78f96c9cf497369d009323c2bb2c894433c3edb0170aaf028d31a180",
  "final_evaluation_lock_sha256": "edc6aaa57bea1f3dc9a7908e4c43df687ed78ddb87acc3455994b545bb25bbb2",
  "primary_model": {
    "name": "Phase 08 best-balanced EA-TTM",
    "epoch": 15
  },
  "final_test_interval": [
    "d_1914",
    "d_1941"
  ],
  "test_extreme_rates": {
    "all": 0.05927470364990863,
    "positive": 0.05546432085461275,
    "negative": 0.0038103827952958814
  },
  "overall_metrics": {
    "Weekly Seasonal Naive": {
      "mse": 7.1657662933983035,
      "rmse": 2.6768948977123297,
      "mae": 1.2439722625685237,
      "wape": 0.8621813918900049
    },
    "Zero-shot TTM": {
      "mse": 4.637857293767649,
      "rmse": 2.153568502223147,
    